In [1]:
import sys
sys.path.append('../../Simulate/')

from StreamMethDB import StreamMethDB
from SetMethylation import SetMethylation
from UtilityFunctions import parseCGmap, parseASM
from UtilityFunctions import retrieve_iupac

In [2]:
import os
import re
import warnings
import pandas as pd
import numpy as np
import subprocess

from Bio import SeqIO
from scipy.stats import beta
from tqdm import tqdm
from typing import Dict, List

In [3]:
class StreamWGSIM:
    '''
    stream WGSIM output for bisulfite reads generation
    :param str  sim_cmd : WGSIM commands for simulation
    :param bool pair_end: pair_end or not
    :rtype None
    '''
    def __init__(self, sim_cmd: list = None, pair_end: bool = True):
        self.sim_cmd  = sim_cmd
        self.pair_end = pair_end

    def __iter__(self):
        return self
    

    def __next__(self):
        wgsim = subprocess.Popen(self.sim_cmd, stdout=subprocess.PIPE, universal_newlines=True)
        sim_iter = iter(wgsim.stdout.readline, b'')

        line  = self.get_line(sim_iter) # line is None when EOF
        while line:
            # collect all variant lines on the contig, after that sim_iter points to read lines
            if line == "Contig Variant Start":
                variant_contig, variant_dict = self.collect_variants(sim_iter)
                yield variant_contig, variant_dict

            # collect read pairs
            for collect_flag, read_pair in self.collect_reads(sim_iter):
                if collect_flag: # {1: collect_reads, 0: swith to collect_vars or EOF}
                    yield False, read_pair
                else:
                    line = "Contig Variant Start" if isinstance(read_pair, list) else None
                    break


    def collect_variants(self, sim_iter):
        '''collect variant lines from stdout'''
        variant_dict = {}
        variant_info = {}

        while True:
            line = self.get_line(sim_iter)
            if line == 'Contig Variant End':
                return variant_info['chrom'], variant_dict

            variant_info = self.process_variant_line(line)
            if variant_info['pos']:
                assert variant_info['pos'] not in variant_dict
                variant_dict[variant_info['pos']] = variant_info


    def collect_reads(self, sim_iter):
        '''collect read lines from stdout'''
        skip_flag = not self.pair_end

        while True:
            line  = self.get_line(sim_iter)
            if not line: # EOF
                yield 0, None
            elif line == "Contig Variant Start": # switch to collect variants
                yield 0, []
            else:
                read1 = self.process_read_lines(sim_iter, line = line)
                read2 = self.process_read_lines(sim_iter, skip = skip_flag)
                yield 1, [read1, read2]


    @staticmethod
    def get_line(sim_iter):
        '''receive lines from console'''
        try:
            line = next(sim_iter).strip()
        except StopIteration:
            print("End of output\n")
            return None
        else:
            return line


    @staticmethod
    def process_variant_line(line: str) -> Dict:
        '''parse variant lines'''
        line_split = line.split('\t')

        try:
            chrom, pos, ref, alt, heter_flag = line_split
        except ValueError:
            return dict(chrom=line_split[0], pos = None)
        else:
            heter = heter_flag == '+'
            indel = int(ref == '-') - int(alt == '-') # 1 for ref=='-', -1 for alt=='-', o.w. 0
            offset= indel * max(len(ref), len(alt))
            if indel:
                iupac  = None
            else:
                iupac  = retrieve_iupac(alt)
                alt    = list(set(iupac) - set(ref))[0]
            return dict(chrom=chrom, pos=int(pos), ref=ref, alt=alt,
                        offset=offset, heter=heter, indel=indel, iupac=iupac)


    @staticmethod
    def process_read_lines(sim_iter, line = None, skip = False):
        '''parse read lines'''
        if skip:
            next(sim_iter)
            next(sim_iter)
            next(sim_iter)
            next(sim_iter)
            return None

        if not line:
            line = next(sim_iter).strip()
        # header, seq, comment process
        read_id, pair, flag_pos, flag_mut, flag_indel, qual, cgr = line.split(' ')
        cgr = np.frombuffer(cgr.encode(), dtype=np.int8)
        seq = np.frombuffer(next(sim_iter).strip().encode(), dtype=np.int8)
        _, start, end, cover_pos, n_sub, n_indel, insert_size, inner_dist, ofs= next(sim_iter).strip().split(':')
        ofs = np.fromstring(ofs, dtype=np.int8, sep = ',')
        ctx = np.frombuffer(next(sim_iter).strip().encode(), np.int8)
        return dict(read_id=read_id, pair=int(pair), qual = int(qual),
                    flag_pos=int(flag_pos), flag_mut=int(flag_mut), flag_indel=int(flag_indel),
                    start=int(start), end=int(end), cover_pos=int(cover_pos),
                    n_sub=int(n_sub), n_indel=int(n_indel),
                    insert_size=int(insert_size), inner_dist=int(inner_dist),
                    cgr=cgr, seq=seq, ofs=ofs, ctx=ctx)


In [4]:
ref_fasta = "/home/wbguo/iproject/BSReadSim/test/ref/BSB_test.fa"
cgmap_file= "/home/wbguo/iproject/BSReadSim/temp/data/sim.CGmap.gz"
asm_file  = "/home/wbguo/iproject/BSReadSim/temp/data/sim.asm.gz"

In [5]:
# meth_set = SetMethylation(ref_fasta=ref_fasta, cgmap_file=cgmap_file, asm_file = asm_file,
#                           outdir="/home/wbguo/iproject/BSReadSim/temp/outdir/",
#                           overwrite_db=True, verbose = True)

In [6]:
meth_set = SetMethylation(meth_db_path="/home/wbguo/iproject/BSReadSim/temp/outdir/",
                          outdir="/home/wbguo/iproject/BSReadSim/temp/outdir/",
                          ref_fasta = ref_fasta)

In [7]:
sim_cmd_part = ['/home/wbguo/iproject/BSReadSim/WGSIM/wgsim', 
                '-1', '100', '-2', '100','-e','0.005','-d','400','-s','25',
                '-r','0.1', '-N','1000',
                '-R','0.15','-X','0.15',
                '-S','2022',
                '-A','0.05','-h','0', '-m', '1', ref_fasta]

In [8]:
for contig_id in meth_set.ref_dict.keys():
    sim_cmd  = sim_cmd_part + ['-c', contig_id]
    read_gen = StreamWGSIM(sim_cmd=sim_cmd)
    var_contig, sim_data= next(read_gen)
    break

[wgsim] seed = 2022
[wgsim_core] calculating the total length of the reference sequence...
[wgsim_core] 6 contig sequences, total length: 1961600
[wgsim_core] Simulate 216 reads from contig chr10 (calculate from -N, as -n is not specified)...
[wgsim_core] No VCF input, will generate SNP randomly if mutation rate is nonzero


ValueError: too many values to unpack (expected 2)

In [10]:
read_gen

In [9]:
next(read_gen)

<generator object StreamWGSIM.__next__ at 0x7f8cf0c622e0>

In [13]:
for i in next(read_gen):
    print(i)

[wgsim] seed = 2022
[wgsim_core] calculating the total length of the reference sequence...
[wgsim_core] 6 contig sequences, total length: 1961600
[wgsim_core] Simulate 216 reads from contig chr10 (calculate from -N, as -n is not specified)...
[wgsim_core] No VCF input, will generate SNP randomly if mutation rate is nonzero
IOPub data rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_data_rate_limit`.

Current values:
NotebookApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
NotebookApp.rate_limit_window=3.0 (secs)



(False, [{'read_id': '@chr10:269921:270350:3b', 'pair': 0, 'qual': 56, 'flag_pos': 1, 'flag_mut': 61440, 'flag_indel': 1, 'start': 269920, 'end': 270019, 'cover_pos': 1, 'n_sub': 4, 'n_indel': 1, 'insert_size': 429, 'inner_dist': 229, 'cgr': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=int8), 'seq': array([2, 3, 3, 3, 3, 0, 2, 2, 2, 0, 0, 1, 0, 2, 3, 2, 3, 0, 1, 0, 1, 2,
       0, 3, 2, 3, 2, 1, 0, 2, 2, 1, 0, 1, 0, 1, 1, 0, 2, 3, 0, 0, 3, 1,
       1, 1, 0, 2, 1, 0, 1, 3, 3, 3, 2, 2, 0, 1, 2, 2, 1, 1, 2, 0, 2, 0,
       3, 2, 2, 2, 1, 0, 2, 0, 3, 1, 0, 1, 1, 3, 0, 0, 1, 1, 0, 3, 0, 0,
       0, 0, 1, 3, 0, 1, 3, 1, 1, 1, 1, 0], dtype=int8), 'ofs': array([ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,

(False, [{'read_id': '@chr10:317381:317774:7b', 'pair': 0, 'qual': 56, 'flag_pos': 1, 'flag_mut': 61440, 'flag_indel': 1, 'start': 317380, 'end': 317479, 'cover_pos': 1, 'n_sub': 5, 'n_indel': 1, 'insert_size': 393, 'inner_dist': 195, 'cgr': array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=int8), 'seq': array([1, 3, 1, 3, 3, 1, 0, 1, 1, 0, 2, 0, 2, 1, 0, 0, 0, 2, 0, 1, 0, 2,
       2, 3, 2, 0, 0, 0, 0, 0, 2, 0, 1, 0, 3, 3, 0, 3, 2, 3, 3, 2, 2, 1,
       3, 2, 2, 2, 3, 2, 1, 1, 2, 3, 2, 2, 1, 3, 1, 0, 3, 2, 1, 1, 3, 2,
       3, 0, 0, 3, 1, 3, 3, 0, 2, 3, 3, 2, 2, 0, 3, 3, 1, 0, 2, 2, 3, 3,
       3, 3, 2, 3, 3, 2, 3, 3, 3, 1, 3, 1], dtype=int8), 'ofs': array([ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,

[wgsim_core] Generated 216 read pairs, with 216 contain SNP, 184 contain INDEL


(False, [{'read_id': '@chr10:131132:131558:be', 'pair': 0, 'qual': 56, 'flag_pos': 1, 'flag_mut': 61440, 'flag_indel': 1, 'start': 131131, 'end': 131231, 'cover_pos': 1, 'n_sub': 7, 'n_indel': 0, 'insert_size': 426, 'inner_dist': 225, 'cgr': array([0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0,
       0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=int8), 'seq': array([1, 0, 2, 1, 0, 0, 3, 3, 1, 3, 1, 1, 0, 3, 1, 3, 1, 0, 0, 1, 3, 1,
       3, 1, 3, 3, 3, 2, 1, 1, 0, 0, 1, 0, 3, 2, 0, 0, 0, 1, 3, 1, 0, 0,
       1, 3, 3, 2, 0, 0, 1, 3, 1, 0, 1, 3, 1, 0, 0, 0, 0, 2, 3, 0, 3, 0,
       0, 0, 2, 3, 1, 3, 2, 2, 0, 0, 0, 1, 0, 1, 3, 1, 3, 2, 3, 3, 3, 2,
       3, 0, 0, 0, 2, 3, 1, 3, 2, 1, 0, 2], dtype=int8), 'ofs': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [ ]:
print(read_gen)

In [ ]:
print(next(read_gen))

In [ ]:
count = 0
for i in read_gen:
    print(i)
    count +=1
    if count ==2:
        break

In [ ]:
i